In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import eigs
import time


def run_comprehensive_analysis(u_c, k_opt, n_bins=5000):
    # 配置参数
    steps_auto = 2*10**9      # 自治系统步数
    steps_non_auto = 2*10**9  # 非自治系统步数
    k_eig = 500             # 提取特征值数量
    
    def get_spectrum(u_func, steps):
        x = 0.5
        # 热启动：消除初态影响
        for _ in range(100000):
            u = u_func(_)
            x = 1 - u * x**2
            
        orbit = np.zeros(steps)
        for i in range(steps):
            u = u_func(i)
            x = 1 - u * x**2
            orbit[i] = x
            
        # 构造 P-F 矩阵
        bins = ((orbit + 1.0) / 2.0 * (n_bins - 1)).astype(int)
        bins = np.clip(bins, 0, n_bins - 1)
        P = csr_matrix((np.ones(len(bins)-1), (bins[:-1], bins[1:])), shape=(n_bins, n_bins))
        
        # 归一化
        row_sums = np.array(P.sum(axis=1)).flatten()
        row_sums[row_sums == 0] = 1.0
        P = P.multiply(1.0 / row_sums[:, np.newaxis])
        
        # 提取特征值相位
        vals, _ = eigs(P, k=k_eig, which='LM', tol=1e-5)
        phases = np.angle(vals[np.abs(vals) > 0.6])
        phases = np.sort(phases[phases > 0.05])
        return phases

    # 1. 执行自治系统分析 (k=0)
    print("正在运行自治系统统计 (10^8 steps)...")
    phases_auto = get_spectrum(lambda n: u_c, steps_auto)
    s_auto = np.diff(phases_auto) / np.mean(np.diff(phases_auto))
    std_auto = np.std(s_auto)

    # 2. 执行非自治系统校准 (k=12.32)
    print(f"正在运行非自治位置校准 (k={k_opt})...")
    # 老化项公式：u_n = u_c - k/ln(n)^2
    # 为了避免 n=0,1 的错误，从一个较大的偏移开始
    n_offset = 10**6 
    phases_non_auto = get_spectrum(lambda n: u_c - k_opt * (np.log(n + n_offset))**-2, steps_non_auto)
    
    # 3. 绘图对比
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

    # --- 左图：GUE 统计验证 ---
    ax1.hist(s_auto, bins=40, density=True, alpha=0.6, color='blue', label=f'Autonomous ($\sigma$={std_auto:.4f})')
    s_theory = np.linspace(0, 4, 100)
    p_gue = (32 / np.pi**2) * (s_theory**2) * np.exp(-4 * s_theory**2 / np.pi)
    ax1.plot(s_theory, p_gue, 'r-', lw=3, label='Standard GUE')
    ax1.set_title('Spectral Statistics (The Fingerprint)', fontsize=14)
    ax1.set_xlabel('Normalized Spacing $s$')
    ax1.legend()

    # --- 右图：位置对齐校准 ---
    TRUE_GAMMAS = [14.1347, 21.0220, 25.0108, 30.4248, 32.9350]
    # 对齐第一个零点
    scaling = TRUE_GAMMAS[0] / phases_non_auto[0]
    sim_gammas = phases_non_auto[:5] * scaling
    
    ax2.vlines(TRUE_GAMMAS, 0, 1, colors='red', linestyles='--', label='Riemann Zeros ($\gamma_n$)')
    ax2.vlines(sim_gammas, 0, 0.8, colors='blue', linestyles='-', label='Non-Auto Spectrum (k=12.32)')
    ax2.set_title('Position Alignment (The Lock-in)', fontsize=14)
    ax2.set_xlabel('Energy Level / Phase')
    ax2.set_ylim(0, 1.2)
    ax2.legend()

    plt.tight_layout()
    plt.savefig('genius_paper_results.png')
    plt.show()

    # 打印数值结果
    print(f"\n自治系统标准差: {std_auto:.6f} (与 GUE 0.422 相比)")
    print("\n前 5 模式位置校准结果:")
    for i, (sim, true) in enumerate(zip(sim_gammas, TRUE_GAMMAS)):
        print(f"模式 {i+1}: 模拟={sim:.4f} | 真实={true:.4f} | 误差={sim-true:.4f}")

# 执行
u_c_fixed = 1.543689012
k_optimum = 12.32

t0 = time.time()
run_comprehensive_analysis(u_c_fixed, k_optimum)
print(f"执行时间：{time.time() - t0:.2f} 秒")

<>:60: SyntaxWarning: invalid escape sequence '\s'
<>:74: SyntaxWarning: invalid escape sequence '\g'
<>:60: SyntaxWarning: invalid escape sequence '\s'
<>:74: SyntaxWarning: invalid escape sequence '\g'
/tmp/ipykernel_3363/702232931.py:60: SyntaxWarning: invalid escape sequence '\s'
  ax1.hist(s_auto, bins=40, density=True, alpha=0.6, color='blue', label=f'Autonomous ($\sigma$={std_auto:.4f})')
/tmp/ipykernel_3363/702232931.py:74: SyntaxWarning: invalid escape sequence '\g'
  ax2.vlines(TRUE_GAMMAS, 0, 1, colors='red', linestyles='--', label='Riemann Zeros ($\gamma_n$)')


正在运行自治系统统计 (10^8 steps)...
